# D1 · Reading a one-hot: names, memory and the dummy variable trap

**Type**: Interpret · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Interpret the output of a `OneHotEncoder` (categories, names, memory usage), show with the rank of the matrix
that with the intercept the columns are redundant, and read the coefficients of a regression with `drop="first"`.

## Dataset

`ds.loans(with_missing=False)`: the two nominals `city` (12 levels) and `contract_type` (4), 1500 rows, and the target
`defaulted` (0/1) as the variable to model with a linear regression (linear probability model: the coefficients
are differences in default rates).

## Educational purpose

Understand that the full one-hot is **linearly dependent** with the intercept (the sum of the columns of a feature is 1) and
that `drop="first"` does not change the predictions but **changes the meaning of the coefficients**: they are differences with respect to a reference
category.

## Task

1. With `enc = OneHotEncoder(sparse=False)` on the two columns save `n_columns` (number of columns produced), `names` (list of the names) and `baseline = enc.categories_[1][0]`, the first category of `contract_type` (the one that `drop="first"` would remove).
2. With `OneHotEncoder(sparse=True)` (default) save `nnz` (nonzero elements) and `density = nnz / (rows · columns)`. Why is it exactly 2 / n_columns?
3. With the full matrix add the column of ones: `A = np.column_stack([np.ones(n), X_ohe])`. Save `rank_without_drop` (`np.linalg.matrix_rank(A)`) and `columns_of_A`. Then redo it with `drop="first"`: `rank_with_drop` and `columns_with_drop`.
4. Train `LinearRegression()` on `y = defaulted` with the full one-hot and with `drop="first"`. Save `max_coef_full` and `max_coef_drop` (maximum absolute value of the coefficients in the two models) and `pred_diff` (maximum absolute difference between the predictions of the two models on the training set).
5. With **only** `contract_type` and `drop="first"`, train the regression and save `unemployed_coef` (the coefficient of `contract_type_unemployed`) and `mean_diff`: the difference between the mean default rate of the unemployed and that of the `baseline` category, computed with `groupby`.
6. Assign to `true_false` `{"a", "b", "c"}`:
   - **a**: with `drop="first"` the coefficient of a category is the effect with respect to the dropped category, not with respect to the mean;
   - **b**: changing the dropped category changes the predictions of the model;
   - **c**: with the full one-hot and the intercept, an unregularized OLS can produce enormous, meaningless coefficients, even though the predictions stay almost correct.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

df = ds.loans(with_missing=False)
X_cat = df[["city", "contract_type"]]
y = df["defaulted"]
n = len(df)

## Your work

In [ ]:
enc = OneHotEncoder(sparse=False).fit(X_cat)
n_columns = ...
names = ...
baseline = ...

In [ ]:
X_sparse = OneHotEncoder().fit_transform(X_cat)
nnz = ...
density = ...

In [ ]:
X_ohe = enc.transform(X_cat)
A = np.column_stack([np.ones(n), X_ohe])
rank_without_drop, columns_of_A = ..., ...

enc_d = OneHotEncoder(sparse=False, drop="first").fit(X_cat)
A_d = np.column_stack([np.ones(n), enc_d.transform(X_cat)])
rank_with_drop, columns_with_drop = ..., ...

In [ ]:
max_coef_full = ...
max_coef_drop = ...
pred_diff = ...

In [ ]:
enc_c = OneHotEncoder(sparse=False, drop="first").fit(df[["contract_type"]])
model = LinearRegression().fit(enc_c.transform(df[["contract_type"]]), y)
unemployed_coef = ...
mean_diff = ...

In [ ]:
true_false = {"a": ..., "b": ..., "c": ...}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`enc.get_feature_names_out()` for the names. `X_sparse.nnz` for the nonzero elements. The predictions: `LinearRegression().fit(A[:, 1:], y).predict(A[:, 1:])` against the same with `A_d`. The difference between means: `df.groupby("contract_type")["defaulted"].mean()` and then `m["unemployed"] - m[baseline]`.'):
    check(n_columns == 16 and len(names) == 16 and baseline == "fixed_term", "16 columns, and the alphabetical baseline is 'fixed_term'",
          "12 cities + 4 contracts = 16; categories_[1][0] is the first sorted category of contract_type")
    check(nnz == 2 * n and close(density, 2 / 16), "the one-hot is very sparse: 2 nonzero values per row, density 2/16",
          "every row has exactly one 1 per feature, so 2 nonzero out of 16 columns")
    check(columns_of_A == 17 and rank_without_drop == 15, "without drop the rank is 15 out of 17 columns: two linear dependencies",
          "one dependency per feature (the sum of its columns is the column of ones): 17 - 2 = 15")
    check(columns_with_drop == 15 and rank_with_drop == 15, "with drop='first' the 15 columns are independent (full rank)",
          "with drop='first' 1 + 11 + 3 = 15 columns remain, all independent")
    check(max_coef_full > 100 * max_coef_drop, "with the full one-hot the coefficients explode (over 100 times those with drop)",
          "without drop the OLS is ill-conditioned: the maximum of the coefficients must be huge compared to the one with drop")
    check(pred_diff < 0.1, "the predictions are almost equal (the same model, apart from numerical instability)",
          "the column space is the same: the predictions must differ little (less than 0.1)")
    check(abs(unemployed_coef - mean_diff) < 1e-9,
          "the coefficient matches the difference between the means of the two groups", "with a single dummy feature the coefficient is the difference of means")
    check(true_false == {"a": True, "b": False, "c": True}, "statements judged correctly",
          "changing the reference category changes the coefficients, not the predictions")
done()

---
The commented solution is in `solutions/D_interpret/D1_reading_a_one_hot_and_the_dummy_trap.ipynb`. Open it after you pass the check, or after you have really tried.